# VisionAid — Exploratory Data Analysis

**ENSF 444 Final Project**

This notebook explores the pre-computed landmark feature dataset before any model is trained.
All features were extracted from the [dsmlr/faceshape](https://github.com/dsmlr/faceshape) image dataset
using OpenCV face detection and the LBF facemark model.

## How to run this notebook

1. Complete the project setup (see README or `run_project.py` for instructions).
2. Ensure `data/processed/landmark_features.csv` exists.  If it does not, run:
   ```
   python run_project.py --force-features
   ```
3. Launch Jupyter from the project root:
   ```
   jupyter notebook notebooks/
   ```
4. Run all cells with **Kernel → Restart & Run All**.

## Contents

1. Dataset loading and inspection
2. Class distribution
3. Feature statistics per class
4. PCA projection
5. Geometry feature correlation heatmap
6. Per-class geometry feature distributions

## 1. Setup and dataset loading

In [ ]:
# Standard library and third-party imports
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Add the project src/ directory to the path so we can import visionaid
# (This is only needed when running from the notebooks/ subdirectory)
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from visionaid.data_loader import (
    load_features,
    get_feature_columns,
    get_geometry_columns,
    get_landmark_columns,
    dataset_summary,
    CLASSES,
    LABEL_COLUMN,
)

# Use a consistent plot style throughout this notebook
sns.set_theme(style="whitegrid", palette="viridis")
plt.rcParams["figure.dpi"] = 120

In [ ]:
# Load the pre-computed landmark feature CSV.
# The CSV path is relative to the project root, so we pass an explicit path.
CSV_PATH = PROJECT_ROOT / "data" / "processed" / "landmark_features.csv"

df = load_features(csv_path=CSV_PATH)

# Quick sanity check — shape should be approximately (493, 162)
print(f"Loaded DataFrame: {df.shape[0]} rows × {df.shape[1]} columns")

In [ ]:
# Print a structured summary of the dataset
dataset_summary(df)

In [ ]:
# Preview the first few rows — metadata columns plus a sample of feature columns
display_cols = ["label", "image_path", "face_w", "face_h"] + get_geometry_columns(df)[:4]
df[display_cols].head()

## 2. Class distribution

The dataset contains 500 labeled images split across five face-shape classes.
A balanced distribution means accuracy and weighted F1 are both reliable metrics.

In [ ]:
# Count images per class
class_counts = df[LABEL_COLUMN].value_counts().sort_index()
print("Images per class:")
print(class_counts.to_string())
print(f"\nTotal processed: {len(df)}  (dropped: {500 - len(df)})")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

# Bar chart — one bar per class, sorted alphabetically
sns.countplot(
    data=df,
    x=LABEL_COLUMN,
    order=sorted(df[LABEL_COLUMN].unique()),
    hue=LABEL_COLUMN,
    palette="viridis",
    legend=False,
    ax=ax,
)
ax.set_title("Class Distribution", fontsize=14)
ax.set_xlabel("Face Shape")
ax.set_ylabel("Image Count")

# Annotate each bar with its count
for patch in ax.patches:
    count = int(patch.get_height())
    ax.text(
        patch.get_x() + patch.get_width() / 2,
        count + 0.5,
        str(count),
        ha="center",
        va="bottom",
        fontsize=11,
    )

plt.tight_layout()
plt.show()

## 3. Feature statistics per class

Looking at the mean value of each geometry feature grouped by class gives an
intuitive sense of which features might separate the classes.

In [ ]:
geom_cols = get_geometry_columns(df)

# Compute mean geometry feature values per class
class_means = df.groupby(LABEL_COLUMN)[geom_cols].mean().round(4)

print("Mean geometry feature values per face-shape class:")
display(class_means.T)  # transposed for readability

In [ ]:
# Visualize selected geometry features with box plots to see class separation
# These four were identified as the most informative by the Random Forest
selected_geom = [
    "geom_jaw_angle",
    "geom_cheekbone_to_height",
    "geom_jaw_width_to_height",
    "geom_forehead_to_jaw",
]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes_flat = axes.flatten()

for i, col in enumerate(selected_geom):
    sns.boxplot(
        data=df,
        x=LABEL_COLUMN,
        y=col,
        order=sorted(df[LABEL_COLUMN].unique()),
        palette="tab10",
        ax=axes_flat[i],
    )
    # Clean up the feature name for the title
    axes_flat[i].set_title(col.replace("geom_", "").replace("_", " ").title())
    axes_flat[i].set_xlabel("")

fig.suptitle("Top Geometry Features by Face-Shape Class", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 4. PCA projection

Reducing the full feature matrix to two principal components lets us visualize
whether the five classes form separable clusters in feature space.  If they do,
a linear classifier should have a good chance of learning the boundaries.

In [ ]:
feature_cols = get_feature_columns(df)

# Scale features before PCA so that high-variance columns (landmark pixels)
# do not dominate the principal components
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[feature_cols])

# Fit PCA with 2 components for 2-D visualization
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

explained = pca.explained_variance_ratio_
print(f"PC1 explains {explained[0]:.2%} of variance")
print(f"PC2 explains {explained[1]:.2%} of variance")
print(f"Total explained: {explained.sum():.2%}")

In [ ]:
pca_df = pd.DataFrame({
    "PC1": X_pca[:, 0],
    "PC2": X_pca[:, 1],
    "label": df[LABEL_COLUMN],
})

fig, ax = plt.subplots(figsize=(9, 7))
sns.scatterplot(
    data=pca_df,
    x="PC1",
    y="PC2",
    hue="label",
    hue_order=sorted(df[LABEL_COLUMN].unique()),
    palette="tab10",
    s=60,
    alpha=0.75,
    ax=ax,
)
ax.set_title(
    f"PCA Projection — Landmark + Geometry Features\n"
    f"Explained variance: {explained.sum():.2%}",
    fontsize=13,
)
ax.set_xlabel(f"PC1 ({explained[0]:.2%} variance)")
ax.set_ylabel(f"PC2 ({explained[1]:.2%} variance)")
plt.tight_layout()
plt.show()

## 5. Geometry feature correlation heatmap

Highly correlated features are somewhat redundant.  Knowing which pairs
are correlated helps interpret model behaviour and could inform future
feature selection.

In [ ]:
geom_cols = get_geometry_columns(df)

# Compute the Pearson correlation matrix
corr_matrix = df[geom_cols].corr()

# Shorten column names for the heatmap axes
short_names = {col: col.replace("geom_", "") for col in geom_cols}
corr_display = corr_matrix.rename(index=short_names, columns=short_names)

fig, ax = plt.subplots(figsize=(13, 11))
sns.heatmap(
    corr_display,
    cmap="coolwarm",
    center=0,
    square=True,
    annot=True,
    fmt=".2f",
    annot_kws={"size": 8},
    ax=ax,
)
ax.set_title("Pearson Correlation — Engineered Geometry Features", fontsize=13)
plt.tight_layout()
plt.show()

## 6. Per-class landmark position heatmap

Averaging the 68 normalized landmark positions per class reveals the
typical face shape 'template' for each category.

In [ ]:
lm_x_cols = [f"lm_x_{i:02d}" for i in range(68)]
lm_y_cols = [f"lm_y_{i:02d}" for i in range(68)]

fig, axes = plt.subplots(1, len(CLASSES), figsize=(15, 4))

for ax, cls in zip(axes, sorted(CLASSES)):
    cls_df = df[df[LABEL_COLUMN] == cls]

    # Average x and y positions across all images in this class
    mean_x = cls_df[lm_x_cols].mean().values
    mean_y = cls_df[lm_y_cols].mean().values

    # Plot the mean landmark positions as scatter points
    # y is inverted so the plot matches the natural image orientation
    ax.scatter(mean_x, -mean_y, s=10, c="steelblue", alpha=0.8)
    ax.set_title(cls, fontsize=11)
    ax.set_xlim(-0.1, 1.1)
    ax.set_ylim(-1.1, 0.1)
    ax.set_aspect("equal")
    ax.axis("off")

fig.suptitle("Mean Normalized Landmark Positions per Class", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## Summary

Key observations from the EDA:

- The dataset is **nearly balanced** (~99 images per class after dropping 7 failures),
  so accuracy and weighted F1 are both fair summary metrics.
- The **PCA projection** shows partial but imperfect class separation — the classes
  overlap significantly, which explains why test accuracy tops out around 62%.
- The **jaw angle**, **cheekbone-to-height**, and **jaw-width-to-height** features
  show the strongest per-class differences in the box plots.
- Several geometry features are highly correlated (e.g. cheekbone_to_height and
  jaw_width_to_height), suggesting that feature selection could reduce
  dimensionality without losing much predictive power.

Continue to `02_model_training_and_evaluation.ipynb` for model comparison.